# Pointwise upper bounds for the number of shapes

This notebook computes the pointwise upper bounds in Section IV-D of *Asymptotic Growth of the Number of Rubik's Snake Shapes*. It uses `../rubiks_snake.py`, NumPy, Numba, SciPy, the standard library, and a Jupyter runtime. It does not depend on unpublished data or other notebooks.

For the full finite-window graph, $S_n\leq\mathbf1^T A_m^{n-m-1}\mathbf1$. A positive integer vector with $A_mv\leq qv$ gives
$$S_n\leq Hq^{n-m-1}=A_2q^n,\qquad H=\frac{\sum v}{\min v},\quad A_2=Hq^{-(m+1)},\quad n\geq m+1.$$
Here $A_{v,u}=\#(u\to v)$. For a pointwise bound we retain the edges between strongly connected components. Removing them would preserve the spectral radius but change the path counts.

The function finds a candidate vector by floating-point iteration and verifies the integer inequality without overflow. It computes prefactors and bound values as exact fractions. `integer_upper_bound` takes the smaller of the exponential certificate and $4^{n-1}$. Since $S_n$ is an integer, the exact rational upper bound can be rounded down.

The saved outputs were reproduced from these cell sources on 2026-09-19 with the repository interpreter (Python 3.14.6, NumPy 2.5.3, Numba 0.67.0, SciPy 1.18.1). Execution counts are null because validation ran in a script outside the editor kernel. Rerunning reconstructs the graph and certificate vector; timings include both graph construction and verification. The much larger $m=13$ graph is not needed for this bound.

In [ ]:
from pathlib import Path
from time import perf_counter
from fractions import Fraction
import sys
started = perf_counter()
cwd = Path.cwd()
candidates = (cwd, cwd / 'rubiks-snake', cwd.parent, cwd.parent / 'rubiks-snake')
snake_dir = next((p for p in candidates if (p / 'rubiks_snake.py').is_file()), None)
if snake_dir is None:
    raise FileNotFoundError('Run from asymptotic-analysis, rubiks-snake, or the repository root')
sys.path.insert(0, str(snake_dir.resolve()))
import numpy as np
import numba
import scipy
from rubiks_snake import window_upper_bound
print(f'Python {sys.version.split()[0]}; NumPy {np.__version__}; Numba {numba.__version__}; SciPy {scipy.__version__}')
print(f'Setup: {perf_counter() - started:.3f} s')

In [ ]:
def get_bound(m, iterations=500, scale=10**12, denominator=10**9):
    """Return an exactly verified full-graph pointwise bound."""
    started = perf_counter()
    result = window_upper_bound(m, iterations, scale, denominator, full_graph=True)
    q, h = result['bound'], result['pointwise_factor']
    result['first_n'] = m + 1
    result['exponential_factor'] = h / q**(m + 1)
    result['seconds'] = perf_counter() - started
    print(f"m={m}: {result['states']} states, {result['edges']} edges")
    print(f"min(v)={result['vector_min']}, sum(v)={result['vector_sum']}")
    print(f"S_n <= ({h}) * ({q})**(n-{m+1}), n >= {m+1}")
    print(f"Approximate A_2: {float(result['exponential_factor']):.12g}; exact certificate passed; {result['seconds']:.3f} s")
    return result

def integer_upper_bound(result, n):
    if not isinstance(n, int) or n < 1:
        raise ValueError('n must be a positive integer')
    elementary = 4**(n - 1)
    if n < result['first_n']:
        return elementary
    value = result['pointwise_factor'] * result['bound']**(n - result['first_n'])
    return min(elementary, value.numerator // value.denominator)

In [ ]:
SMALL_WINDOWS = (3, 5, 7)
small_results = [get_bound(m) for m in SMALL_WINDOWS]

m=3: 64 states, 241 edges
min(v)=1, sum(v)=57293229651747
S_n <= (57293229651747) * (3810528899/1000000000)**(n-4), n >= 4
Approximate A_2: 271745440344; exact certificate passed; 0.003 s
m=5: 920 states, 3384 edges
min(v)=1, sum(v)=754946603021323
S_n <= (754946603021323) * (1860055621/500000000)**(n-6), n >= 6
Approximate A_2: 284826684763; exact certificate passed; 0.007 s
m=7: 12585 states, 46471 edges
min(v)=1, sum(v)=10232025022770017
S_n <= (10232025022770017) * (370307551/100000000)**(n-8), n >= 8
Approximate A_2: 289375027940; exact certificate passed; 0.061 s


## Published full-graph certificate

Increase `WINDOW` to seek a smaller base, bearing in mind that memory use grows exponentially. The full graph may need more iterations than the SCC calculation. Every returned certificate is valid, even when it gives a weak bound. A smaller base can come with a larger prefactor, so compare the bounds at the lengths you need. To reproduce the saved research vector, use 500 unshifted iterations and scale $10^{12}$.

In [ ]:
WINDOW, ITERATIONS, SCALE = 9, 500, 10**12
published = get_bound(WINDOW, ITERATIONS, SCALE)
if (WINDOW, ITERATIONS, SCALE) == (9, 500, 10**12):
    assert published['bound'] < Fraction(36855, 10000)
    assert published['exponential_factor'] < 294632981756
    print(f"Rounded corollary: S_n < 2.95e11*(3.6855)**n for n >= {published['first_n']}")
started = perf_counter()
for n in (4, 10, 14, 28, 300):
    print(f'n={n}: S_n <= {integer_upper_bound(published, n)}')
print(f'Pointwise evaluations: {perf_counter() - started:.3f} s')

m=9: 172226 states, 633138 edges
min(v)=1, sum(v)=136209763558000711
S_n <= (136209763558000711) * (1842734183/500000000)**(n-10), n >= 10
Approximate A_2: 294632981756; exact certificate passed; 1.135 s
Rounded corollary: S_n < 2.95e11*(3.6855)**n for n >= 10
n=4: S_n <= 64
n=10: S_n <= 262144
n=14: S_n <= 67108864
n=28: S_n <= 18014398509481984
n=300: S_n <= 1037378892220248239628101965922790287753111558060609224998914332422663202853227036599926762236775948572049471652825197295598787768852943826971718708528490921765295450850377380921344
Pointwise evaluations: 0.000 s
